In [1]:
import os
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# Load dataset
data_path = '../generated/data.csv' if os.path.exists('../generated/data.csv') else 'generated/data.csv'
df = pd.read_csv(data_path)

# Display top 30 rows
print(f"Total Dataset Records: {len(df):,} rows | Columns: {len(df.columns)}")
df.head(30)

Total Dataset Records: 5,288 rows | Columns: 15


,demographics.gender,vitals.diastolic_bp,demographics.age,anthropometry.height,labs.glucose,anthropometry.bmi,history.hypertensive,history.family_diabetes,vitals.pulse_rate,vitals.systolic_bp,history.cardiovascular_disease,outcome.diabetic,history.family_hypertension,history.stroke,anthropometry.weight
0,Male,85,43,1.63,7.60,19.27,1,0,76,118,1,No,0,0,51.2
1,Female,98,57,1.54,6.94,22.94,0,0,70,122,0,No,0,0,54.4
2,Male,93,30,1.65,5.73,15.17,0,1,56,165,0,No,1,0,41.3
3,Female,78,37,1.61,3.04,19.79,0,0,50,127,1,No,0,0,51.3
4,Female,100,39,1.45,9.09,30.06,0,0,73,177,0,Yes,1,0,63.2
5,Female,76,49,1.59,8.02,24.25,0,1,95,212,0,Yes,1,0,61.3
6,Male,89,34,1.51,4.47,23.86,0,0,51,120,0,No,0,0,54.4
7,Female,120,35,1.61,7.17,33.02,0,0,45,135,0,No,0,0,85.6
8,Female,110,37,1.61,6.40,22.80,1,0,87,125,0,No,0,0,59.1
9,Male,83,42,1.48,5.37,20.36,1,0,73,160,0,No,0,0,44.6


In [2]:
num_cols = [
    "demographics.age", "anthropometry.height", "anthropometry.weight", "anthropometry.bmi", 
    "vitals.systolic_bp", "vitals.diastolic_bp", "vitals.pulse_rate", "labs.glucose"
]

# Compute descriptive statistics
num_summary = df[num_cols].describe().T
num_summary["median"] = df[num_cols].median()
num_summary["variance"] = df[num_cols].var()
num_summary["iqr"] = df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25)
num_summary["skewness"] = df[num_cols].skew()

# Select & reorder summary metrics
display_cols = ["count", "mean", "std", "variance", "min", "25%", "median", "75%", "max", "iqr", "skewness"]
num_summary = num_summary[display_cols]

# Format index feature names
num_summary.index = [col.split('.')[-1] for col in num_summary.index]
num_summary.index.name = "Feature"
num_summary.round(4)

,count,mean,std,variance,min,25%,median,75%,max,iqr,skewness
Feature,,,,,,,,,,,
age,5288.0,47.0303,13.4443,180.7496,21.00,36.0000,44.000,56.00,80.00,20.0000,0.4841
height,5288.0,1.5399,0.0789,0.0062,1.20,1.5100,1.540,1.59,2.00,0.0800,-0.8492
weight,5288.0,50.8902,9.7272,94.6184,30.00,45.4000,50.500,55.90,102.10,10.5000,0.5588
bmi,5288.0,21.6375,4.7991,23.0313,9.91,18.7875,21.225,24.03,60.35,5.2425,0.9276
systolic_bp,5288.0,142.5108,28.3287,802.5178,82.00,123.0000,134.000,157.00,230.00,34.0000,1.1085
diastolic_bp,5288.0,90.8033,14.0867,198.4345,45.00,82.0000,90.000,99.00,120.00,17.0000,0.2353
pulse_rate,5288.0,72.6224,13.3933,179.3813,40.00,64.0000,72.000,80.00,120.00,16.0000,0.6131
glucose,5288.0,8.1566,4.7155,22.2357,2.00,5.5000,7.020,8.94,30.00,3.4400,2.4076


In [3]:
cat_cols = [
    "demographics.gender", "outcome.diabetic", "history.hypertensive", 
    "history.family_diabetes", "history.cardiovascular_disease", 
    "history.family_hypertension", "history.stroke"
]

for col in cat_cols:
    feature_name = col.split('.')[-1]
    counts = df[col].value_counts(dropna=False)
    pcts = df[col].value_counts(normalize=True, dropna=False) * 100
    freq_df = pd.DataFrame({"Count": counts, "Percentage (%)": pcts.round(2)})
    freq_df.index.name = feature_name
    
    display(HTML(f"<h3>Feature: [{feature_name}]</h3>"))
    display(freq_df)

,Count,Percentage (%)
gender,,
Male,2694,50.95
Female,2594,49.05


,Count,Percentage (%)
diabetic,,
No,4473,84.59
Yes,815,15.41


,Count,Percentage (%)
hypertensive,,
0,4060,76.78
1,1228,23.22


,Count,Percentage (%)
family_diabetes,,
0,4493,84.97
1,795,15.03


,Count,Percentage (%)
cardiovascular_disease,,
0,4975,94.08
1,313,5.92


,Count,Percentage (%)
family_hypertension,,
0,4570,86.42
1,718,13.58


,Count,Percentage (%)
stroke,,
0,5051,95.52
1,237,4.48


In [4]:
# Grouping continuous numerical features by diabetic status ('No' vs 'Yes')
subgroup_stats = df.groupby("outcome.diabetic")[num_cols].agg(["mean", "std", "median"]).T
subgroup_stats.index = subgroup_stats.index.map(lambda x: (x[0].split('.')[-1], x[1]))
subgroup_stats.index.names = ["Feature", "Metric"]
subgroup_stats.round(2)

outcome.diabetic         No     Yes
Feature      Metric                
age          mean     46.86   47.98
             std      13.61   12.47
             median   44.00   47.00
height       mean      1.54    1.54
             std       0.08    0.08
             median    1.54    1.54
weight       mean     50.95   50.59
             std       9.75    9.62
             median   50.60   50.40
bmi          mean     21.65   21.56
             std       4.79    4.84
             median   21.22   21.24
systolic_bp  mean    142.25  143.95
             std      28.12   29.42
             median  134.00  135.00
diastolic_bp mean     90.79   90.90
             std      14.21   13.39
             median   90.00   90.00
pulse_rate   mean     72.47   73.45
             std      13.30   13.90
             median   71.00   72.00
glucose      mean      8.11    8.42
             std       4.67    4.97
             median    6.97    7.19

In [5]:
# Frequency counts crosstab
ctab_counts = pd.crosstab(df["demographics.gender"], df["outcome.diabetic"], margins=True, margins_name="Total")
ctab_counts.index.name = "Gender"
display(HTML("<h3>Counts Table</h3>"))
display(ctab_counts)

# Row percentages crosstab
ctab_pct = pd.crosstab(df["demographics.gender"], df["outcome.diabetic"], normalize="index") * 100
ctab_pct.index.name = "Gender"
display(HTML("<h3>Row Percentages (%) Table</h3>"))
display(ctab_pct.round(2))

outcome.diabetic,No,Yes,Total
Gender,,,
Female,2224,370,2594
Male,2249,445,2694
Total,4473,815,5288


outcome.diabetic,No,Yes
Gender,,
Female,85.74,14.26
Male,83.48,16.52
